In [79]:
import pandas as pd
from IPython.display import display
import numpy as np
import re
import os

df_hist = pd.read_excel(r'C:\Users\a.vorona\Desktop\Work\Dingli\Отчет остатки-обороты по складу ПО ЗЧ.xlsx', skiprows=[0, 2], usecols=[0, 4, 7, 8, 9, 10])
df = pd.read_excel('final_модели.xlsx')

In [80]:
def join_ordered(series):
    vals = series.dropna().astype(str).str.strip()
    vals = [v for v in vals if v]
    seen = set()
    result = []
    for v in vals:
        if v not in seen:
            seen.add(v)
            result.append(v)
    return "; ".join(result) if result else None

def merge_parts(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["Артикул "] = df["Артикул "].astype(str) 

    agg_map = {
        "Расход": "sum",
        'Конечный остаток': 'last',
        'Номенклатура': join_ordered
    }

    df_agg = df.groupby("Артикул ", as_index=False).agg(agg_map)
    df_agg['Расход'] = df_agg['Расход'].astype(int)
    df_agg['Конечный остаток'] = df_agg['Конечный остаток'].fillna(0)
    df_agg['Конечный остаток'] = df_agg['Конечный остаток'].astype(int)

    return df_agg

In [81]:
df_merged = merge_parts(df_hist)

df_merged['Артикул чистый'] = (
    df_merged['Артикул ']
    .astype(str)
    .str.extract(r'(\d+)', expand=False)
)

df_filtered = df_merged.merge(
    df[['Part Number', 'Description', 'Models']],
    left_on='Артикул чистый',
    right_on='Part Number',
    how='inner'
)

df_filtered

,Артикул,Расход,Конечный остаток,Номенклатура,Артикул чистый,Part Number,Description,Models
0,00000267A,0,1,Палец 00000267A,00000267,00000267,892-20045 Pin,JCPT0808; JCPT0812; JCPT1008; JCPT1012; JCPT12...
1,00000364A,0,8,Втлука пальца рулевого BA16-18NE 00000364A (88...,00000364,00000364,8812-242820 Sleeve Bearing; 8812-242820 Bearing,BA16NE; BA18NE; JCPT1523RTB; JCPT1823RTB
2,00000368A,0,4,Втулка 00000368A,00000368,00000368,8812-252820 Bearing,AMWP11.5-8200AC; JCPT0608DCS; JCPT0708DCS; JCP...
3,00000390A,0,8,Втулка 00000390A,00000390,00000390,8812-323645 Bearing,JCPT0808; JCPT0812; JCPT1008; JCPT1012; JCPT12...
4,00000419A,0,4,Втулка 00000419A,00000419,00000419,8813-405040 Sleeve Bearing,BA16NE; BA18NE; BA16CERT2; BA18CERT2; BA20CERT...
...,...,...,...,...,...,...,...,...
401,93300721,5,12,Кронштейн датчика угла 93300721,93300721,93300721,Bracket,BA36RT; BA41RT; BA44RT
402,93300854A,0,3,Датчик давления 93300854,93300854,93300854,"Pressure Sensor; Sensor; Sensor, Pressure Gauge",BA36RT; BA41RT; BA44RT
403,93301069,0,2,Палец 93301069,93301069,93301069,Pin,BA36RT; BA41RT; BA44RT
404,НФ-00001065,0,21,Кабель питания MF-14,00001065,00001065,250A Fuse,JCPT0808; JCPT1008; JCPT1012; JCPT1212; JCPT14...


In [86]:
df_filtered['Сколько закупать'] = np.maximum(0, df_filtered['Расход']-df_filtered['Конечный остаток'])
df_filtered = df_filtered[
    ~df_filtered['Артикул ']
    .astype(str)
    .str.startswith('НФ-')
]

In [88]:
df_filtered['Description'] = (
    df_filtered['Description']
    .str.replace(r'\(.*?\)', '', regex=True)
    .str.strip()
)

def normalize_unique(text):
    if pd.isna(text):
        return text
    
    parts = [p.strip() for p in text.split(';')]
    parts = [p for p in parts if p]  
    
    seen = set()
    result = []
    for p in parts:
        if p not in seen:
            seen.add(p)
            result.append(p)
            
    return '; '.join(result)

df_filtered['Description'] = df_filtered['Description'].apply(normalize_unique)

df_filtered.to_excel('forecast.xlsx', index=False)